# BdSLW60 - Sequence-network baselines on the official split

In [1]:
# ---- Cell 1: config and paths ----
import os, re, json, time, math, random, collections, subprocess, sys, shutil, hashlib, warnings
from pathlib import Path
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore', message='undefined skeleton graph')
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning, module='torch.nn.init')

SESSION_START = time.time()        # the 12-hour Kaggle limit counts from here (roughly)
WORK = Path('/kaggle/working')
CACHE = WORK / 'bdslw60_w2'
KP_DIR = CACHE / 'keypoints'
for d in (CACHE, KP_DIR):
    d.mkdir(parents=True, exist_ok=True)
INPUT_ROOT = Path('/kaggle/input')

# Week-1 keypoints: found automatically (a folder named 'keypoints' holding U*W*.npz files). Set by hand if needed.
KP_SOURCE = "/kaggle/input/notebooks/rishta19/bdslw60-week-1-v2/bdslw60_cache/keypoints"
if KP_SOURCE is None:
    cands = sorted({p.parent for p in INPUT_ROOT.rglob('U*W*.npz') if p.parent.name == 'keypoints'},
                   key=lambda d: -len(list(d.glob('*.npz'))))
    KP_SOURCE = cands[0] if cands else None
print('week-1 keypoints:', KP_SOURCE, '(%d files)' % len(list(Path(KP_SOURCE).glob('*.npz'))) if KP_SOURCE else '')
assert KP_SOURCE is not None, 'Add the week-1 v2 notebook output as an input (it holds bdslw60_cache/keypoints).'

T_OUT = 64
N_NODES = 49
MAX_GAP = 5
TEST_SIGNERS = (4, 8)            # never evaluated before week 3
P1_VAL_SIGNER = 5
INNER_VAL_SIGNERS = (1, 10, 13)
MIN_VAL_CLIPS = 500
SLGCN_REPO = 'https://github.com/jackyjsy/CVPR21Chal-SLR.git'
SLGCN_COMMIT = 'de6c53a6fbbc08a5d9a728e33614a4541836714c'   # pinned so results are reproducible

WORD_IDS = (list(range(1, 13)) + [19, 20] + list(range(37, 51)) + list(range(91, 101))
            + [111, 112] + list(range(211, 221)) + list(range(351, 361)))
assert len(WORD_IDS) == 60
WORD2CLS = {w: i for i, w in enumerate(WORD_IDS)}
TABLE6 = {1: 975, 2: 744, 3: 583, 4: 637, 5: 600, 6: 815, 7: 107, 8: 639, 9: 656,
          10: 383, 11: 771, 12: 655, 13: 621, 14: 112, 15: 655, 16: 97, 17: 116, 18: 141}
VID_RE = re.compile(r'U(\d+)W(\d+)([A-Za-z]*)', re.I)
all_files = [p for p in INPUT_ROOT.rglob('*') if p.is_file() and 'bdslw60' in str(p).lower()]
print('config ok')


week-1 keypoints: /kaggle/input/notebooks/rishta19/bdslw60-week-1-v2/bdslw60_cache/keypoints (770 files)
config ok


In [2]:
# ---- Cell 2: official SL-GCN code (pinned commit) + the 49-node BdSL graph file ----
REPO_DIR = WORK / 'CVPR21Chal-SLR'
if not (REPO_DIR / 'SL-GCN').exists():
    subprocess.run(['git', 'clone', '-q', SLGCN_REPO, str(REPO_DIR)], check=True)
subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', '-q', SLGCN_COMMIT], check=True)
SLGCN_DIR = REPO_DIR / 'SL-GCN'

# BdSL graph: 7 body nodes (nose, shoulders, elbows, wrists) + 21 left-hand + 21 right-hand nodes.
HAND_EDGES = [(0, 1), (1, 2), (2, 3), (3, 4), (0, 5), (5, 6), (6, 7), (7, 8), (5, 9), (9, 10), (10, 11), (11, 12),
              (9, 13), (13, 14), (14, 15), (15, 16), (13, 17), (17, 18), (18, 19), (19, 20), (0, 17)]
BODY_EDGES = [(0, 1), (0, 2), (1, 2), (1, 3), (2, 4), (3, 5), (4, 6)]
EDGES = (BODY_EDGES + [(a + 7, b + 7) for a, b in HAND_EDGES] + [(a + 28, b + 28) for a, b in HAND_EDGES]
         + [(5, 7), (6, 28)])

def build_parents(edges, V=N_NODES, root=0):
    adj = collections.defaultdict(list)
    for a, b in edges:
        adj[a].append(b); adj[b].append(a)
    parent = [-1] * V
    parent[root] = root
    queue = [root]
    while queue:
        u = queue.pop(0)
        for w in adj[u]:
            if parent[w] == -1:
                parent[w] = u
                queue.append(w)
    assert all(p >= 0 for p in parent), 'graph is not connected'
    return np.array(parent)

PARENT = build_parents(EDGES)
INWARD = [(int(v), int(PARENT[v])) for v in range(N_NODES) if PARENT[v] != v]   # child -> parent, as in sign_27.py
GRAPH_FILE = """import sys
sys.path.extend(['../'])
from graph import tools

num_node = %d
self_link = [(i, i) for i in range(num_node)]
inward = %r
outward = [(j, i) for (i, j) in inward]
neighbor = inward + outward


class Graph:
    def __init__(self, labeling_mode='spatial'):
        self.num_node = num_node
        self.self_link = self_link
        self.inward = inward
        self.outward = outward
        self.neighbor = neighbor
        self.A = tools.get_spatial_graph(num_node, self_link, inward, outward)
""" % (N_NODES, INWARD)
(SLGCN_DIR / 'graph' / 'bdsl49.py').write_text(GRAPH_FILE)

if str(SLGCN_DIR) not in sys.path:
    sys.path.insert(0, str(SLGCN_DIR))
from model.decouple_gcn_attn import Model as SLGCN, TCN_GCN_unit, unit_gcn
from graph.bdsl49 import Graph as BdslGraph
GRAPH_A = BdslGraph('spatial').A                   # (3, V, V): self, inward, outward
assert GRAPH_A.shape == (3, N_NODES, N_NODES)
print('SL-GCN code loaded from', SLGCN_DIR, '| commit', SLGCN_COMMIT[:8], '| graph edges:', len(INWARD))


SL-GCN code loaded from /kaggle/working/CVPR21Chal-SLR/SL-GCN | commit de6c53a6 | graph edges: 48


In [3]:
# ---- Cell 3: videos + annotations -> one table of trials (same parser as week 1, plus the FL fix) ----
rows = []
for p in all_files:
    if p.suffix.lower() == '.mp4':
        m = VID_RE.search(p.stem)
        if m:
            rows.append(dict(path=str(p), stem=p.stem, user=int(m[1]), word=int(m[2]), view=m[3].upper()))
vids_all = pd.DataFrame(rows)
# FIX 3: when several front videos share a (user, word) (e.g. U1W357F_1, _2, _3), use the one week 1 used
# (its keypoints are in KP_SOURCE), so the annotated frame numbers refer to the right recording.
vids_all['in_week1'] = vids_all.stem.map(lambda s: (Path(KP_SOURCE) / (s + '.npz')).exists())
vids_all = vids_all.sort_values(['in_week1', 'stem'], ascending=[False, True]).reset_index(drop=True)
for (u, w), g in vids_all[vids_all.view == 'F'].groupby(['user', 'word']):
    if len(g) > 1:
        print('U%dW%d has %d front videos: using %s, not %s' % (u, w, len(g), g.stem.iloc[0], g.stem.iloc[1:].tolist()))
front = vids_all[vids_all.view == 'F'].drop_duplicates(['user', 'word'])
# FIX 1: a non-front video (e.g. U18W358FL) is used only when that (user, word) has no front-view video at all
have_front = set(zip(front.user, front.word))
extra = vids_all[(vids_all.view != 'F') & ~vids_all.apply(lambda r: (r.user, r.word) in have_front, axis=1)]
extra = extra.drop_duplicates(['user', 'word'])
for r in extra.itertuples():
    print('using non-front video because no front copy exists:', r.stem)
for r in vids_all[(vids_all.view != 'F') & ~vids_all.stem.isin(extra.stem)].itertuples():
    print('not used (a front copy exists):', r.stem)
videos = pd.concat([front, extra]).reset_index(drop=True)
videos = videos[videos.word.isin(WORD_IDS)].reset_index(drop=True)
USE_VIEWS = {(r.user, r.word): r.view for r in videos.itertuples()}
print('videos used:', len(videos))

json_files = sorted(p for p in all_files if p.suffix.lower() == '.json' and 'output' in p.name.lower())

def iter_dicts(obj, path=()):
    if isinstance(obj, dict):
        yield path, obj
        for k, v in obj.items():
            yield from iter_dicts(v, path + (str(k),))
    elif isinstance(obj, list):
        for i, v in enumerate(obj):
            yield from iter_dicts(v, path + ('[%d]' % i,))

def as_int(v):
    try:
        return int(float(v))
    except Exception:
        return None

segs = []
for jp in json_files:
    with open(jp, encoding='utf-8') as f:
        data = json.load(f)
    for path, d in iter_dicts(data):
        if 'starting' not in d or 'ending' not in d:
            continue
        sig = ' '.join(list(path) + [str(v) for v in d.values() if isinstance(v, (str, int))])
        m = VID_RE.search(sig)
        if m:
            user, word, view = int(m[1]), int(m[2]), m[3].upper()
        else:                                             # fallback: user and word from the tree levels (W1 -> U1 -> ...)
            pu = [int(x[1:]) for x in path if re.fullmatch(r'U\d+', x)]
            pw = [int(x[1:]) for x in path if re.fullmatch(r'W\d+', x)]
            if not pu or not pw:
                continue
            user, word = pu[-1], pw[-1]
            view = USE_VIEWS.get((user, word), '')
        # week 1 kept only view '' or 'F'; now any view is kept if it is the video we use for this (user, word)
        if view not in ('', 'F') and USE_VIEWS.get((user, word)) != view:
            continue
        hand = next((p for p in path if p in ('LeftHand', 'RightHand')), None)
        starts, ends = d['starting'], d['ending']
        if not isinstance(starts, list):
            starts, ends = [starts], [ends]
        for s, e in zip(starts, ends):
            s, e = as_int(s), as_int(e)
            if s is None or e is None or e <= s:
                continue
            segs.append(dict(user=user, word=word, start=s, end=e, hand='L' if hand == 'LeftHand' else 'R', src=jp.name))
segs = pd.DataFrame(segs).drop_duplicates(['user', 'word', 'start', 'end'])
assert segs.hand.notna().all()
trials = segs.merge(videos[['user', 'word', 'stem', 'path']], on=['user', 'word'], how='inner')
trials['cls'] = trials.word.map(WORD2CLS)
trials = trials.sort_values(['stem', 'start']).reset_index(drop=True)
cmp = pd.DataFrame({'ours': trials.groupby('user').size(), 'paper': pd.Series(TABLE6)}).fillna(0).astype(int)
cmp['diff'] = cmp.ours - cmp.paper
print('trials:', len(trials), '(paper 9307; week 1 had 9297)')
print(cmp.T.to_string())


U1W357 has 3 front videos: using U1W357F_2, not ['U1W357F_1', 'U1W357F_3']
using non-front video because no front copy exists: U18W358FL
videos used: 775
trials: 9300 (paper 9307; week 1 had 9297)
        1    2    3    4    5    6    7    8    9    10   11   12   13   14   15  16   17   18
ours   970  742  583  637  600  815  107  639  656  383  771  655  621  112  655  97  116  141
paper  975  744  583  637  600  815  107  639  656  383  771  655  621  112  655  97  116  141
diff    -5   -2    0    0    0    0    0    0    0    0    0    0    0    0    0   0    0    0


In [4]:
# ---- Cell 4: keypoints - copy the week-1 files, extract only what is missing (e.g. the FL video) ----
need = trials.groupby('stem').agg(path=('path', 'first'), f0=('start', 'min'), f1=('end', 'max')).reset_index()
n_copied = 0
for r in need.itertuples():
    src, dst = Path(KP_SOURCE) / (r.stem + '.npz'), KP_DIR / (r.stem + '.npz')
    if src.exists() and not dst.exists():
        shutil.copy(src, dst)
        n_copied += 1
todo = [r for r in need.itertuples() if not (KP_DIR / (r.stem + '.npz')).exists()]
print('copied %d keypoint files; %d video(s) still to extract: %s' % (n_copied, len(todo), [r.stem for r in todo]))

if todo:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'mediapipe'], check=True)
    import urllib.request, cv2
    import mediapipe as mp
    from mediapipe.tasks import python as mp_python
    from mediapipe.tasks.python import vision as mp_vision
    model_path = WORK / 'holistic_landmarker.task'
    if not model_path.exists():
        found = list(INPUT_ROOT.rglob('holistic_landmarker.task'))
        if found:
            shutil.copy(found[0], model_path)
        else:
            urllib.request.urlretrieve('https://storage.googleapis.com/mediapipe-models/holistic_landmarker/'
                                       'holistic_landmarker/float16/latest/holistic_landmarker.task', model_path)
    POSE_IDX = [0, 11, 12, 13, 14, 15, 16]
    for r in todo:                                   # same settings as the week-1 extractor
        f0, f1 = max(0, int(r.f0) - 5), int(r.f1) + 5
        opts = mp_vision.HolisticLandmarkerOptions(
            base_options=mp_python.BaseOptions(model_asset_path=str(model_path)),
            running_mode=mp_vision.RunningMode.VIDEO,
            min_pose_detection_confidence=0.5, min_pose_landmarks_confidence=0.5,
            min_hand_landmarks_confidence=0.5, min_face_detection_confidence=0.5,
            min_face_landmarks_confidence=0.5)
        h = mp_vision.HolisticLandmarker.create_from_options(opts)
        cap = cv2.VideoCapture(r.path)
        for _ in range(f0):
            if not cap.grab():
                break
        coords, masks = [], []
        for fi in range(f0, f1 + 1):
            ok, frame = cap.read()
            if not ok:
                break
            H, W = frame.shape[:2]
            if max(H, W) > 640:
                s = 640.0 / max(H, W)
                frame = cv2.resize(frame, (int(W * s), int(H * s)))
            rgb = np.ascontiguousarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            res = h.detect_for_video(mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb), (fi - f0) * 33)
            c = np.zeros((49, 3), np.float32); m = np.zeros(49, np.uint8)
            if res.pose_landmarks:
                for k, idx in enumerate(POSE_IDX):
                    lm = res.pose_landmarks[idx]
                    c[k] = (lm.x, lm.y, lm.z); m[k] = 1
            for hand, sl, wrist in ((res.left_hand_landmarks, slice(7, 28), 5), (res.right_hand_landmarks, slice(28, 49), 6)):
                if hand:
                    arr = np.array([[p.x, p.y, p.z] for p in hand], np.float32)
                    if m[wrist]:
                        arr[:, 2] += c[wrist, 2]
                    c[sl] = arr; m[sl] = 1
            coords.append(c); masks.append(m)
        h.close(); cap.release()
        np.savez_compressed(KP_DIR / (r.stem + '.npz'), coords=np.stack(coords), mask=np.stack(masks), f0=f0)
        print('extracted', r.stem, len(coords), 'frames')
print('keypoint files ready:', len(list(KP_DIR.glob('*.npz'))))


copied 770 keypoint files; 1 video(s) still to extract: ['U18W358FL']


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1793295602.493622     103 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1793295602.530467     106 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1793295602.536851     103 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1793295602.537959     106 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature infer

extracted U18W358FL 632 frames
keypoint files ready: 771


In [5]:
# ---- Cell 5: preprocessing functions + self-tests on synthetic data ----
LEFT_BODY, RIGHT_BODY = [1, 3, 5], [2, 4, 6]
PERM = np.arange(N_NODES)
for l, r in zip(LEFT_BODY, RIGHT_BODY):
    PERM[l], PERM[r] = r, l
PERM[7:28], PERM[28:49] = np.arange(28, 49), np.arange(7, 28)

def fill_gaps(coords, mask, max_gap=MAX_GAP):
    # linear interpolation of gaps up to max_gap frames that are bounded by detections on both sides
    coords, mask = coords.copy(), mask.copy()
    for n in range(coords.shape[1]):
        idx = np.flatnonzero(mask[:, n])
        for a, b in zip(idx[:-1], idx[1:]):
            gap = b - a - 1
            if 1 <= gap <= max_gap:
                w = ((np.arange(a + 1, b) - a) / (b - a))[:, None]
                coords[a + 1:b, n] = (1 - w) * coords[a, n] + w * coords[b, n]
                mask[a + 1:b, n] = 1
    return coords, mask

def normalize(coords, mask):
    # centre on mid-shoulder, divide by shoulder width, per frame; missing nodes stay exactly 0
    ls, rs = coords[:, 1], coords[:, 2]
    ok = (mask[:, 1] > 0) & (mask[:, 2] > 0)
    if not ok.any():
        return None, None
    mid = (ls + rs) / 2
    width = np.linalg.norm((ls - rs)[:, :2], axis=1)
    good = ok & (width > 1e-4)
    if not good.any():
        return None, None
    mid_f = np.where(ok[:, None], mid, np.median(mid[ok], axis=0))
    width_f = np.where(good, width, np.median(width[good]))
    out = (coords - mid_f[:, None, :]) / width_f[:, None, None]
    out = out * mask[..., None]
    return out.astype(np.float32), mask

def mirror(coords, mask):
    c = coords[:, PERM].copy()
    c[..., 0] *= -1
    return c, mask[:, PERM]

def resample(coords, mask, t_out=T_OUT):
    # mask-weighted linear interpolation so a detected frame never blends with a zero-filled neighbour
    T = len(coords)
    if T == 1:
        return np.repeat(coords, t_out, 0), np.repeat(mask, t_out, 0)
    src = np.linspace(0, T - 1, t_out)
    lo = np.floor(src).astype(int)
    hi = np.minimum(lo + 1, T - 1)
    w = (src - lo)
    wv, wc = w[:, None], w[:, None, None]
    mf = mask.astype(np.float32)
    num = (1 - wc) * (coords[lo] * mf[lo][..., None]) + wc * (coords[hi] * mf[hi][..., None])
    den = (1 - wv) * mf[lo] + wv * mf[hi]
    mask_out = mask[np.rint(src).astype(int)]
    coords_out = np.where(mask_out[..., None] > 0, num / np.maximum(den[..., None], 1e-6), 0.0)
    return coords_out.astype(np.float32), mask_out

# ---- self-tests ----
rng = np.random.default_rng(0)
c = rng.normal(size=(30, N_NODES, 3)).astype(np.float32)
c[:, 1] = [0.6, 0.5, 0.0]; c[:, 2] = [0.4, 0.5, 0.0]          # shoulders: width 0.2
m = np.ones((30, N_NODES), np.uint8)
m[5:8, 10] = 0; m[12:20, 11] = 0; m[0:3, 12] = 0              # short gap, long gap, leading gap
c = c * m[..., None]
c2, m2 = fill_gaps(c, m)
assert m2[5:8, 10].all() and not m2[12:20, 11].any() and not m2[0:3, 12].any()
n, nm = normalize(c2, m2)
assert np.allclose(np.abs(n[:, 1, 0]), 0.5, atol=1e-4)      # shoulders end up at +-0.5
assert np.all(n[nm == 0] == 0)
mc, mm = mirror(n, nm)
mc2, mm2 = mirror(mc, mm)
assert np.allclose(mc2, n) and (mm2 == nm).all()
assert np.allclose(mc[:, 1, 0], -n[:, 2, 0])                 # left/right shoulders swapped and mirrored
const = np.ones((20, N_NODES, 3), np.float32); cm = np.ones((20, N_NODES), np.uint8)
cm[8:14, 20] = 0; const = const * cm[..., None]
rc, rm = resample(const, cm)
assert rc.shape == (T_OUT, N_NODES, 3) and rm.shape == (T_OUT, N_NODES)
assert np.allclose(rc[rm[:, 20] > 0, 20], 1.0)               # no partial values at mask borders
assert np.all(rc[rm == 0] == 0)
print('preprocessing self-tests passed')

preprocessing self-tests passed


In [6]:
# ---- Cell 6: build the arrays (B0 inputs: zero-filled + detection mask), drop duplicate trials ----
def build(trials):
    n = len(trials)
    X = np.zeros((n, 3, T_OUT, N_NODES), np.float32)
    M = np.zeros((n, T_OUT, N_NODES), np.uint8)
    ok = np.zeros(n, bool)
    hand_missing_raw = np.full(n, np.nan, np.float32)
    cur_stem, kp = None, None
    for i, r in enumerate(trials.itertuples()):
        if r.stem != cur_stem:
            f = KP_DIR / (r.stem + '.npz')
            kp = None
            if f.exists():
                z = np.load(f)
                kp = dict(coords=z['coords'], mask=z['mask'], f0=int(z['f0']))
            cur_stem = r.stem
        if kp is None:
            continue
        s, e = r.start - kp['f0'], r.end - kp['f0']
        if s < 0 or e >= len(kp['coords']) or e - s < 2:
            continue
        c, m = kp['coords'][s:e + 1], kp['mask'][s:e + 1]
        hand_missing_raw[i] = (m[:, 7:] == 0).mean()          # before any processing, for the missing-rate analysis
        c, m = normalize(c, m)
        if c is None:
            continue
        if r.hand == 'L':
            c, m = mirror(c, m)
        c, m = resample(c, m)
        X[i] = c.transpose(2, 0, 1); M[i] = m; ok[i] = True
    return X, M, ok, hand_missing_raw

t0 = time.time()
X, M, ok, hm = build(trials)
meta = trials[ok].reset_index(drop=True)
X, M = X[ok], M[ok]
meta['hand_missing'] = hm[ok]
print('built in %.0f s | usable trials %d / %d' % (time.time() - t0, len(meta), len(trials)))

# FIX 2: identical trials within one signer (double annotations) - keep the first copy
h = [hashlib.md5(np.round(X[i], 4).tobytes()).hexdigest() for i in range(len(X))]
meta['hash'] = h
dup = meta.duplicated(['user', 'hash'], keep='first').values
print('identical duplicate trials removed:', int(dup.sum()))
for r in meta[dup].itertuples():
    print('   ', r.stem, r.start, r.end)
cross = meta[~dup].groupby('hash').user.nunique()
assert (cross <= 1).all(), 'identical trials under different signers - check before training'
X, M, meta = X[~dup], M[~dup], meta[~dup].reset_index(drop=True)
y = meta.cls.values.astype(np.int64)
np.save(CACHE / 'X.npy', X); np.save(CACHE / 'M.npy', M); np.save(CACHE / 'y.npy', y)
meta.to_csv(CACHE / 'meta.csv', index=False)
cmp = pd.DataFrame({'ours': meta.groupby('user').size(), 'paper': pd.Series(TABLE6)}).fillna(0).astype(int)
cmp['diff'] = cmp.ours - cmp.paper
print('\nfinal trials per signer vs Table 6:')
print(cmp.T.to_string())
print('total %d | test U4+U8 %d (paper 1276) | classes %d' % (len(meta), meta.user.isin(TEST_SIGNERS).sum(), len(set(y))))


built in 9 s | usable trials 9299 / 9300
identical duplicate trials removed: 0

final trials per signer vs Table 6:
        1    2    3    4    5    6    7    8    9    10   11   12   13   14   15  16   17   18
ours   969  742  583  637  600  815  107  639  656  383  771  655  621  112  655  97  116  141
paper  975  744  583  637  600  815  107  639  656  383  771  655  621  112  655  97  116  141
diff    -6   -2    0    0    0    0    0    0    0    0    0    0    0    0    0   0    0    0
total 9299 | test U4+U8 1276 (paper 1276) | classes 60


In [7]:
# ---- Cell 7: split files (P1 official split, inner model-selection folds, 18 leave-one-signer-out folds) ----
users = sorted(int(u) for u in meta.user.unique())
clips_per_user = meta.groupby('user').size().to_dict()
lh_per_user = meta[meta.hand == 'L'].groupby('user').size().reindex(users, fill_value=0).to_dict()
def idx_of(us):
    return np.flatnonzero(meta.user.isin(us).values)

# P1: final runs. Train 15 signers, pick the checkpoint on U5, test on U4+U8 once (week 3 only).
p1 = {'train': idx_of([u for u in users if u not in TEST_SIGNERS + (P1_VAL_SIGNER,)]),
      'val': idx_of([P1_VAL_SIGNER]), 'test': idx_of(list(TEST_SIGNERS))}
print('P1 sizes -> train %d | val (U%d) %d | test (U4+U8) %d   (paper/plan: ~7431 / 600 / 1276)'
      % (len(p1['train']), P1_VAL_SIGNER, len(p1['val']), len(p1['test'])))

# Inner folds: all model decisions (B0 vs B1 vs B2 ...). Each holds out one of INNER_VAL_SIGNERS and trains
# on every other non-test signer. U4/U8 appear in none of them.
inner = {}
for v in INNER_VAL_SIGNERS:
    tr_users = [u for u in users if u not in TEST_SIGNERS + (v,)]
    inner[v] = {'train': idx_of(tr_users), 'val': idx_of([v])}
    assert not set(meta.user.values[inner[v]['train']]) & set(TEST_SIGNERS + (v,))
    print('inner fold val U%-2d: train %d | val %d (left-handed %d)'
          % (v, len(inner[v]['train']), len(inner[v]['val']), lh_per_user[v]))

# LOSO: a TEST-TIME protocol (two of its folds test on U4 and U8). Run it in week 3 only.
# Each fold's validation signer comes from the signers with >= MIN_VAL_CLIPS clips.
big = [u for u in users if clips_per_user[u] >= MIN_VAL_CLIPS]
loso = {}
for k, s in enumerate(users):
    cands = [u for u in big if u != s]
    v = cands[k % len(cands)]
    loso[s] = {'train': idx_of([u for u in users if u not in (s, v)]), 'val': idx_of([v]), 'test': idx_of([s])}
print('LOSO validation signers (test -> val):', {s: int(meta.user.values[d['val'][0]]) for s, d in loso.items()})
assert all(len(d['val']) >= MIN_VAL_CLIPS for d in loso.values())

splits = {'P1': {k: v.tolist() for k, v in p1.items()},
          'INNER': {str(v): {k: a.tolist() for k, a in d.items()} for v, d in inner.items()},
          'LOSO': {str(s): {k: v.tolist() for k, v in d.items()} for s, d in loso.items()}}
json.dump(splits, open(CACHE / 'splits.json', 'w'))
print('LOSO folds written:', len(loso))
print('files in cache:', sorted(p.name for p in CACHE.iterdir() if p.is_file()))


P1 sizes -> train 7423 | val (U5) 600 | test (U4+U8) 1276   (paper/plan: ~7431 / 600 / 1276)
inner fold val U1 : train 7054 | val 969 (left-handed 648)
inner fold val U10: train 7640 | val 383 (left-handed 148)
inner fold val U13: train 7402 | val 621 (left-handed 186)
LOSO validation signers (test -> val): {1: 2, 2: 3, 3: 4, 4: 5, 5: 6, 6: 8, 7: 8, 8: 11, 9: 12, 10: 12, 11: 15, 12: 1, 13: 2, 14: 2, 15: 4, 16: 4, 17: 5, 18: 6}
LOSO folds written: 18
files in cache: ['M.npy', 'X.npy', 'meta.csv', 'splits.json', 'y.npy']


In [8]:
# ---- Cell 8: augmentation and bone/motion streams (graph and PARENT come from cell 2) ----
def time_warp(x, m, gamma):
    # Monotonic warp that keeps the whole sign: frame t samples the clip at (T-1) * (t/(T-1))**gamma.
    # gamma < 1: the start is played slower and the end faster; gamma > 1: the opposite. First and last frames stay.
    C, T, V = x.shape
    src = (T - 1) * np.linspace(0, 1, T) ** gamma
    lo = np.floor(src).astype(int)
    hi = np.minimum(lo + 1, T - 1)
    w = src - lo
    mf = m.astype(np.float32)
    num = (1 - w)[None, :, None] * x[:, lo] * mf[lo][None] + w[None, :, None] * x[:, hi] * mf[hi][None]
    den = (1 - w)[:, None] * mf[lo] + w[:, None] * mf[hi]
    m_out = m[np.rint(src).astype(int)]
    x_out = np.where(m_out[None] > 0, num / np.maximum(den[None], 1e-6), 0.0)
    return x_out.astype(np.float32), m_out

def augment(x, m, rng, p_rot=0.5, p_scale=0.5, p_speed=0.5, p_hdrop=0.2):
    # x: (C, T, V) coordinates, m: (T, V) detection mask. Training only.
    C, T, V = x.shape
    x, m = x.copy(), m.copy()
    if rng.random() < p_speed:                       # rhythm change; keeps the whole sign (see time_warp)
        x, m = time_warp(x, m, float(np.exp(rng.uniform(np.log(0.8), np.log(1.25)))))
    if rng.random() < p_rot:                         # rotation +-15 degrees in the image plane
        th = np.deg2rad(rng.uniform(-15, 15))
        c, s_ = np.cos(th), np.sin(th)
        x0, x1 = x[0].copy(), x[1].copy()
        x[0], x[1] = c * x0 - s_ * x1, s_ * x0 + c * x1
    if rng.random() < p_scale:                       # scale 0.8-1.2
        x = x * rng.uniform(0.8, 1.2)
    if rng.random() < p_hdrop:                       # one hand disappears for a short span
        sl = slice(7, 28) if rng.random() < 0.5 else slice(28, 49)
        L = int(rng.integers(8, 25))
        st = int(rng.integers(0, max(1, T - L)))
        x[:, st:st + L, sl] = 0
        m[st:st + L, sl] = 0
    return (x * m[None]).astype(np.float32), m

def make_stream(x, m, stream, use_mask, use_z=True):
    # returns features (C', T, V) and the matching mask (T, V)
    if stream == 'joint':
        f, fm = x, m
    elif stream == 'bone':
        f = x - x[:, :, PARENT]
        fm = m * m[:, PARENT]
    elif stream == 'joint_motion':
        f = np.zeros_like(x); fm = np.zeros_like(m)
        f[:, :-1] = x[:, 1:] - x[:, :-1]
        fm[:-1] = m[1:] * m[:-1]
    elif stream == 'bone_motion':
        b, bm = make_stream(x, m, 'bone', False)
        f = np.zeros_like(b); fm = np.zeros_like(bm)
        f[:, :-1] = b[:, 1:] - b[:, :-1]
        fm[:-1] = bm[1:] * bm[:-1]
    else:
        raise ValueError(stream)
    if use_mask:
        f = f * fm[None]
    if not use_z:
        f = f[:2]
    return f.astype(np.float32), fm

# ---- self-tests ----
assert PARENT[0] == 0 and len(set(PARENT.tolist())) > 10
_rng = np.random.default_rng(1)
_x = _rng.normal(size=(3, T_OUT, N_NODES)).astype(np.float32)
_m = (_rng.random((T_OUT, N_NODES)) > 0.3).astype(np.uint8)
_x = _x * _m[None]
for _s in ('joint', 'bone', 'joint_motion', 'bone_motion'):
    for _um in (False, True):
        _f, _fm = make_stream(_x, _m, _s, _um)
        assert _f.shape == (3, T_OUT, N_NODES) and _fm.shape == (T_OUT, N_NODES)
        if _um:
            assert np.all(_f[:, _fm == 0] == 0)
_w, _wm = time_warp(_x, _m, 0.8)
assert _w.shape == _x.shape and (_wm[0] == _m[0]).all() and (_wm[-1] == _m[-1]).all()   # whole sign kept
assert np.allclose(time_warp(_x, _m, 1.0)[0], _x)                                       # gamma 1 = identity
_a, _am = augment(_x, _m, np.random.default_rng(2), 1, 1, 1, 1)
assert _a.shape == _x.shape and np.all(_a[:, _am == 0] == 0)
print('augmentation and stream self-tests passed')

augmentation and stream self-tests passed


In [9]:
# ---- Cell 9: models - official SL-GCN (S0), + missing-aware blocks (S2), + signer-adversarial head (S3) ----
import torch, torch.nn as nn, torch.nn.functional as F
from sklearn.metrics import f1_score
device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.backends.cudnn.benchmark = True              # fixed input sizes: let cuDNN pick the fastest kernels

# fp32 islands: these parts overflow or lose accuracy in fp16, so they always run in fp32, even under autocast.
from model.dropSke import DropBlock_Ske
from model.dropT import DropBlockT_1d

def _autocast_on():
    try:
        return torch.is_autocast_enabled('cuda')
    except TypeError:                               # older torch
        return torch.is_autocast_enabled()

def fp32_island(fn):
    if getattr(fn, '_fp32_island', False):
        return fn
    def wrapped(self, x, *args, **kw):
        if torch.is_tensor(x) and x.is_cuda and _autocast_on():   # l1 passes a plain 0 (no residual)
            with torch.autocast(device_type='cuda', enabled=False):
                args = tuple(a.float() if torch.is_tensor(a) and a.is_floating_point() else a for a in args)
                return fn(self, x.float(), *args, **kw)   # every tensor input in fp32 (e.g. the mask)
        return fn(self, x, *args, **kw)
    wrapped._fp32_island = True
    return wrapped

for _cls in (DropBlock_Ske, DropBlockT_1d, unit_gcn):   # patched once; the official classes are otherwise unchanged
    _cls.forward = fp32_island(_cls.forward)
print('device:', device)
assert device == 'cuda', 'The official SL-GCN code creates some tensors on CUDA: turn on the GPU.'

A_BAR = torch.tensor(np.abs(GRAPH_A).sum(0), dtype=torch.float32)   # (V, V) fixed graph, self links included
A_HOP = (A_BAR > 0).float()

class MissingAwareGCN(nn.Module):
    """Wraps an official unit_gcn. Missing nodes (mask 0) are replaced by a learned token, send no messages to
    their neighbours (they still keep their own state through the self link), and each node's output is rescaled
    by the share of its graph neighbourhood that was observed (as in partial convolution)."""
    def __init__(self, g, in_channels):
        super().__init__()
        self.g = g
        self.token = nn.Parameter(torch.zeros(1, in_channels, 1, N_NODES))
        self.register_buffer('A_bar', A_BAR.clone())

    def forward(self, x0, m):                             # x0: (N, C, T, V), m: (N, T, V) float in {0, 1}
        g = self.g
        mm = m[:, None]                                   # (N, 1, T, V)
        xt = x0 * mm + (1 - mm) * self.token
        learn_A = g.DecoupleA.repeat(1, g.out_channels // g.groups, 1, 1)
        A = torch.cat([g.norm(learn_A[k:k + 1]) for k in range(3)], 0)          # (3, C', V, V), source v -> target w
        x = torch.einsum('nctw,cd->ndtw', xt, g.Linear_weight).contiguous() + g.Linear_bias
        x = g.bn0(x)
        n, kc, t, v = x.size()
        x = x.view(n, g.num_subset, kc // g.num_subset, t, v)
        m5 = m[:, None, None]                             # (N, 1, 1, T, V)
        y = torch.einsum('nkctv,kcvw->nctw', x * m5, A)   # messages from observed nodes only
        diag = torch.diagonal(A, dim1=-2, dim2=-1)        # (3, C', V): a missing node keeps its own (token) state
        y = y + torch.einsum('nkctw,kcw->nctw', x * (1 - m5), diag)
        tot = self.A_bar.sum(0)                           # (V,) full neighbourhood weight
        obs = torch.einsum('ntv,vw->ntw', m, self.A_bar) + torch.diagonal(self.A_bar) * (1 - m)
        y = y * (tot / obs.clamp(min=1e-3))[:, None]
        y = g.bn(y)
        y = y + g.down(xt)
        return g.relu(y)

MissingAwareGCN.forward = fp32_island(MissingAwareGCN.forward)

def propagate_mask(m):                                    # a node counts as informed if any 1-hop node was observed
    with torch.autocast(device_type='cuda', enabled=False):   # keep the mask in fp32
        return torch.clamp(torch.einsum('ntv,vw->ntw', m.float(), A_HOP.to(m.device)), max=1.0)

class GradReverse(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, lam):
        ctx.lam = lam
        return x.view_as(x)
    @staticmethod
    def backward(ctx, g):
        return -ctx.lam * g, None

class BdslNet(nn.Module):
    """n_ma = 0, masked_pool False, n_signers 0  ->  exactly the official SL-GCN forward pass (S0)."""
    def __init__(self, n_cls=60, in_ch=3, n_ma=0, masked_pool=False, n_signers=0, groups=16, block_size=41):
        super().__init__()
        self.base = SLGCN(num_class=n_cls, num_point=N_NODES, num_person=1, groups=groups, block_size=block_size,
                          graph='graph.bdsl49.Graph', graph_args={'labeling_mode': 'spatial'}, in_channels=in_ch)
        self.layers = [getattr(self.base, 'l%d' % i) for i in range(1, 11)]
        self.n_ma, self.masked_pool = n_ma, masked_pool
        chans = [in_ch, 64, 64, 64, 64, 128, 128, 128, 256, 256]
        self.ma = nn.ModuleList([MissingAwareGCN(self.layers[i].gcn1, chans[i]) for i in range(n_ma)])
        self.signer_head = (nn.Sequential(nn.Linear(256, 128), nn.ReLU(), nn.Linear(128, n_signers))
                            if n_signers else None)

    def _unit(self, i, x, keep_prob, m):                  # official TCN_GCN_unit.forward, with an optional MA gcn
        u = self.layers[i]
        y = self.ma[i](x, m) if i < self.n_ma else u.gcn1(x)
        if u.attention:
            se = y.mean(-2); se1 = u.sigmoid(u.conv_sa(se)); y = y * se1.unsqueeze(-2) + y
            se = y.mean(-1); se1 = u.sigmoid(u.conv_ta(se)); y = y * se1.unsqueeze(-1) + y
            se = y.mean(-1).mean(-1); se1 = u.relu(u.fc1c(se)); se2 = u.sigmoid(u.fc2c(se1))
            y = y * se2.unsqueeze(-1).unsqueeze(-1) + y
        y = u.tcn1(y, keep_prob, u.A)
        x_skip = u.dropT_skip(u.dropSke(u.residual(x), keep_prob, u.A), keep_prob)
        return u.relu(y + x_skip)

    def forward(self, x, m=None, keep_prob=0.9, lam=0.0):  # x: (N, C, T, V), m: (N, T, V)
        N, C, T, V = x.shape
        b = self.base
        x = b.data_bn(x.permute(0, 3, 1, 2).reshape(N, V * C, T)).view(N, V, C, T).permute(0, 2, 3, 1).contiguous()
        m = torch.ones(N, T, V, device=x.device) if m is None else m.float()
        kps = [1.0] * 6 + [keep_prob] * 4                # the official schedule: DropGraph only in blocks 7-10
        for i in range(10):
            x = self._unit(i, x, kps[i], m)
            if i < self.n_ma:
                m = propagate_mask(m)
        if self.masked_pool:
            w = F.adaptive_max_pool2d(m[:, None], x.shape[2:])
            feat = (x * w).sum(dim=(2, 3)) / w.sum(dim=(2, 3)).clamp(min=1.0)
        else:
            feat = x.mean(dim=(2, 3))
        logits = b.fc(feat)
        if self.signer_head is not None and self.training and lam > 0:
            return logits, self.signer_head(GradReverse.apply(feat, lam))
        return logits, None

# ---- self-tests ----
torch.manual_seed(0)
_g = unit_gcn(3, 64, GRAPH_A, 16, N_NODES).to(device).eval()
_ma = MissingAwareGCN(_g, 3).to(device).eval()
_x = torch.randn(2, 3, 16, N_NODES, device=device)
with torch.no_grad():
    assert torch.allclose(_ma(_x, torch.ones(2, 16, N_NODES, device=device)), _g(_x), atol=1e-4), 'MA layer != SL-GCN when nothing is missing'
    _m = torch.ones(2, 16, N_NODES, device=device); _m[:, :, 7:28] = 0
    _a = _ma(_x, _m); _x2 = _x.clone(); _x2[:, :, :, 7:28] = 123.0
    assert torch.allclose(_a, _ma(_x2, _m), atol=1e-4), 'missing values leaked into the output'
_s0 = BdslNet().to(device).eval()
with torch.no_grad():
    _xx = torch.randn(2, 3, T_OUT, N_NODES, device=device)
    assert torch.allclose(_s0(_xx)[0], _s0.base(_xx[..., None], 1.0), atol=1e-4), 'wrapper != official SL-GCN forward'
if device == 'cuda':                                   # the overflow your teammate found, now contained
    _d = DropBlockT_1d(block_size=41).train()
    _h = torch.randn(64, 256, 16, N_NODES, device='cuda').half()
    with torch.autocast(device_type='cuda', dtype=torch.float16):
        assert torch.isfinite(_d(_h, 0.9)).all(), 'DropBlockT still overflows under autocast'
print('self-tests passed: MA = SL-GCN when nothing is missing; missing values cannot leak; S0 = official forward;'
      ' DropBlock is finite under fp16 autocast')


# ---- compact ST-GCN (the week-1 model), used as a cheap graph baseline ----
def build_A(edges, V=N_NODES):
    A = np.eye(V, dtype=np.float32)
    for a, b in edges:
        A[a, b] = A[b, a] = 1
    d = A.sum(1) ** -0.5
    return (d[:, None] * A * d[None, :]).astype(np.float32)
A_NORM = build_A(EDGES)

class STBlock(nn.Module):
    def __init__(self, cin, cout, A, stride=1, dropout=0.2):
        super().__init__()
        self.register_buffer('A', torch.tensor(A, dtype=torch.float32))
        self.B = nn.Parameter(torch.zeros(A.shape, dtype=torch.float32))   # learnable adjacency offset
        self.gcn = nn.Conv2d(cin, cout, 1)
        self.bn1 = nn.BatchNorm2d(cout)
        self.tcn = nn.Sequential(nn.Conv2d(cout, cout, (9, 1), (stride, 1), (4, 0)),
                                 nn.BatchNorm2d(cout), nn.Dropout(dropout))
        if cin == cout and stride == 1:
            self.res = nn.Identity()
        else:
            self.res = nn.Sequential(nn.Conv2d(cin, cout, 1, (stride, 1)), nn.BatchNorm2d(cout))
    def forward(self, x):                                   # x: (N, C, T, V)
        y = self.gcn(x)
        y = torch.einsum('nctv,vw->nctw', y, self.A + self.B)
        y = F.relu(self.bn1(y))
        y = self.tcn(y)
        return F.relu(y + self.res(x))

class GCNNet(nn.Module):
    def __init__(self, in_ch, n_cls, A, V=N_NODES, widths=(64, 64, 128, 128, 256, 256), strides=(1, 1, 2, 1, 2, 1),
                 masked_pool=False):
        super().__init__()
        self.masked_pool = masked_pool
        self.data_bn = nn.BatchNorm1d(in_ch * V)
        blocks, c = [], in_ch
        for w, s in zip(widths, strides):
            blocks.append(STBlock(c, w, A, s))
            c = w
        self.blocks = nn.ModuleList(blocks)
        self.drop = nn.Dropout(0.3)
        self.fc = nn.Linear(c, n_cls)
    def forward(self, x, mask=None):                        # mask: (N, T, V), 1 = detected
        N, C, T, V = x.shape
        x = self.data_bn(x.permute(0, 1, 3, 2).reshape(N, C * V, T))
        x = x.reshape(N, C, V, T).permute(0, 1, 3, 2).contiguous()
        for b in self.blocks:
            x = b(x)
        if self.masked_pool and mask is not None:          # average only over detected nodes and frames
            w = F.adaptive_max_pool2d(mask[:, None].float(), x.shape[2:])   # (N, 1, T', V)
            pooled = (x * w).sum(dim=(2, 3)) / w.sum(dim=(2, 3)).clamp(min=1.0)
        else:
            pooled = x.mean(dim=(2, 3))
        return self.fc(self.drop(pooled))


class CompactSTGCN(GCNNet):
    """Same interface as BdslNet: forward(x, m, keep_prob, lam) -> (logits, None)."""
    def __init__(self, n_cls=60, in_ch=3):
        super().__init__(in_ch, n_cls, A_NORM)
    def forward(self, x, m=None, keep_prob=0.9, lam=0.0):
        return super().forward(x, m), None

with torch.no_grad():
    assert CompactSTGCN().eval()(torch.randn(2, 3, T_OUT, N_NODES))[0].shape == (2, 60)
print('compact ST-GCN baseline ready')


# ---- the base paper's attention Bi-LSTM (port of pipeline_LSTM_225_FLIPPED_no_stop_B64_D5MAX.ipynb) ----
class BaseBiLSTM(nn.Module):
    """Keras original: bd11 = Dropout(0.3)(BiLSTM(64)(x)); bd2 = BiLSTM(64)(x); att = Attention(use_scale=True)([bd2, bd11]);
    Dense(60, softmax)(Dropout(0.3)(Dense(64, silu)(Flatten()(Concatenate()([bd2, att]))))).
    Keras Attention([query, value]) uses the value as key: softmax(scale * q k^T) v, with a learned scalar scale (init 1)."""
    def __init__(self, n_feat=3 * N_NODES, T=T_OUT, n_cls=60, units=64):
        super().__init__()
        self.in_bn = nn.BatchNorm1d(n_feat)                 # same input normalisation as the graph models' data_bn
        self.lstm1 = nn.LSTM(n_feat, units, batch_first=True, bidirectional=True)
        self.lstm2 = nn.LSTM(n_feat, units, batch_first=True, bidirectional=True)
        self.drop1 = nn.Dropout(0.3)
        self.scale = nn.Parameter(torch.ones(()))
        self.fc1 = nn.Linear(T * 4 * units, 64)
        self.drop2 = nn.Dropout(0.3)
        self.fc2 = nn.Linear(64, n_cls)
    def forward(self, x, m=None, keep_prob=0.9, lam=0.0):  # x: (N, C, T, V) -> per-frame features (N, T, V*C)
        N, C, T, V = x.shape
        f = x.permute(0, 2, 3, 1).reshape(N, T, V * C)
        f = self.in_bn(f.transpose(1, 2)).transpose(1, 2)
        v = self.drop1(self.lstm1(f)[0])                   # value (and key)
        q = self.lstm2(f)[0]                               # query
        w = torch.softmax(self.scale * torch.bmm(q, v.transpose(1, 2)), dim=-1)
        att = torch.bmm(w, v)
        h = torch.cat([q, att], dim=-1).flatten(1)
        return self.fc2(self.drop2(F.silu(self.fc1(h)))), None

with torch.no_grad():
    assert BaseBiLSTM().eval()(torch.randn(2, 3, T_OUT, N_NODES))[0].shape == (2, 60)
print('base-paper Bi-LSTM ready: %.2fM parameters' % (sum(p.numel() for p in BaseBiLSTM().parameters()) / 1e6))


device: cuda


/kaggle/working/CVPR21Chal-SLR/SL-GCN/model/decouple_gcn_attn.py:118: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.eyes = nn.Parameter(torch.tensor(torch.stack(


Attention Enabled!
Attention Enabled!
Attention Enabled!
Attention Enabled!
Attention Enabled!
Attention Enabled!
Attention Enabled!
Attention Enabled!
Attention Enabled!
Attention Enabled!
self-tests passed: MA = SL-GCN when nothing is missing; missing values cannot leak; S0 = official forward; DropBlock is finite under fp16 autocast
compact ST-GCN baseline ready
base-paper Bi-LSTM ready: 1.27M parameters


In [10]:
SEQ_MODELS = {}   # filled in cell 11
# ---- Cell 10: dataset + training loop ----
USE_AMP = True     # mixed precision with fp32 islands (cell 9); cell 10b sets this to False if it is not clean
GRAD_CLIP = 5.0    # same for S0, S2, S3

class BdslDataset(torch.utils.data.Dataset):
    def __init__(self, X, M, y, idx, stream='joint', aug=False, signer_of=None):
        self.X, self.M, self.y, self.idx = X, M, y, np.asarray(idx)
        self.stream, self.aug, self.signer_of = stream, aug, signer_of
    def __len__(self):
        return len(self.idx)
    def __getitem__(self, k):
        i = self.idx[k]
        x, m = np.array(self.X[i]), np.array(self.M[i])
        if self.aug:
            x, m = augment(x, m, np.random.default_rng(int(torch.randint(0, 2 ** 31 - 1, (1,)))))
        f, fm = make_stream(x, m, self.stream, use_mask=True)
        s = self.signer_of[i] if self.signer_of is not None else -1
        return torch.from_numpy(f), torch.from_numpy(fm.astype(np.float32)), int(self.y[i]), int(s)

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

@torch.no_grad()
def predict(model, loader):
    model.eval()
    out, ys = [], []
    for x, m, y_, _ in loader:
        with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=USE_AMP):
            out.append(model(x.to(device), m.to(device))[0].float().cpu())
        ys.append(y_)
    return torch.cat(out), torch.cat(ys)

def scores(logits, ys):
    p = logits.argmax(1)
    return dict(top1=(p == ys).float().mean().item(),
                top5=(logits.topk(5, 1).indices == ys[:, None]).any(1).float().mean().item(),
                f1=f1_score(ys.numpy(), p.numpy(), average='macro'))

def run(name, split, seed, n_ma=0, masked_pool=False, lam_max=0.0, stream='joint', epochs=60, lr=0.1, bs=64,
        warmup=5, tag='', eval_every=5, arch='slgcn', test_idx=None, opt_name='sgd'):
    set_seed(seed)
    Xm = np.load(CACHE / 'X.npy', mmap_mode='r'); Mm = np.load(CACHE / 'M.npy', mmap_mode='r')
    tr_users = sorted(set(meta.user.values[split['train']]))
    sid = {u: k for k, u in enumerate(tr_users)}
    signer_of = np.array([sid.get(u, -1) for u in meta.user.values])
    tr = BdslDataset(Xm, Mm, y, split['train'], stream, aug=True, signer_of=signer_of)
    va = BdslDataset(Xm, Mm, y, split['val'], stream)
    tl = torch.utils.data.DataLoader(tr, batch_size=bs, shuffle=True, num_workers=4, drop_last=True, pin_memory=True)
    vl = torch.utils.data.DataLoader(va, batch_size=128, shuffle=False, num_workers=4)
    if arch in SEQ_MODELS:
        model = SEQ_MODELS[arch]().to(device)
    elif arch == 'stgcn_compact':
        model = CompactSTGCN().to(device)
    else:
        model = BdslNet(n_ma=n_ma, masked_pool=masked_pool, n_signers=len(tr_users) if lam_max > 0 else 0).to(device)
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    params = [p for p in model.parameters() if p.requires_grad]
    if opt_name == 'adamw':
        opt = torch.optim.AdamW(params, lr=lr, weight_decay=0.01)
    else:
        opt = torch.optim.SGD(params, lr=lr, momentum=0.9, nesterov=True, weight_decay=1e-4)
    crit = nn.CrossEntropyLoss(label_smoothing=0.1)
    scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP)
    best = dict(f1=-1)
    evals = []                                   # (epoch, top1, f1) at every validation check
    steps_total, step = epochs * len(tl), 0
    print('%s %s seed %d | n_ma=%d masked_pool=%s lam_max=%.2f | %.2fM params | train %d val %d'
          % (name, tag, seed, n_ma, masked_pool, lam_max, n_params / 1e6, len(tr), len(va)))
    t_start = time.time()
    for ep in range(epochs):
        cur = lr * (ep + 1) / warmup if ep < warmup else 0.5 * lr * (1 + math.cos(math.pi * (ep - warmup) / max(1, epochs - warmup)))
        for gp in opt.param_groups:
            gp['lr'] = cur
        keep_prob = max(0.9, 1.0 - 0.1 * ep / max(1, int(0.4 * epochs)))   # official ramp, scaled to the shorter run
        model.train()
        t0, tot, n = time.time(), 0.0, 0
        for x, m, y_, s_ in tl:
            x, m, y_, s_ = x.to(device), m.to(device), y_.to(device), s_.to(device)
            p = step / steps_total
            lam = lam_max * (2.0 / (1.0 + math.exp(-10 * p)) - 1.0)             # DANN schedule, 0 -> lam_max
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=USE_AMP):
                logits, s_logits = model(x, m, keep_prob=keep_prob, lam=lam)
                loss = crit(logits, y_)
                if s_logits is not None:
                    loss = loss + F.cross_entropy(s_logits, s_)
            if not torch.isfinite(loss):
                raise RuntimeError('loss became %s at epoch %d - lower lr or lam_max' % (loss.item(), ep + 1))
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(params, GRAD_CLIP)
            scaler.step(opt); scaler.update()
            tot += loss.item() * len(y_); n += len(y_); step += 1
        if ep == 0:
            print('  first epoch took %.0f s -> about %.0f min for this run' % (time.time() - t0, (time.time() - t0) * epochs / 60))
        if (ep + 1) % eval_every == 0 or ep == epochs - 1:
            logits_v, ys_v = predict(model, vl)
            sc = scores(logits_v, ys_v)
            evals.append((ep + 1, sc['top1'], sc['f1']))
            if sc['f1'] > best['f1']:
                best = dict(sc, epoch=ep + 1)
            print('  ep %3d | loss %.3f | val top1 %.3f macroF1 %.3f | keep %.2f lam %.3f | %.0f min so far'
                  % (ep + 1, tot / n, sc['top1'], sc['f1'], keep_prob, lam, (time.time() - t_start) / 60))
    last = sc
    pred = pd.DataFrame(dict(idx=np.asarray(split['val']), pred=logits_v.argmax(1).numpy(), true=ys_v.numpy()))
    pred.to_csv(CACHE / ('pred_%s_%s_seed%d.csv' % (name, tag, seed)), index=False)   # last-epoch predictions
    torch.save(model.state_dict(), CACHE / ('%s_%s_seed%d_last.pt' % (name, tag, seed)))
    test = {}
    if test_idx is not None:                      # ONE evaluation on the test signers, after training is finished
        te = BdslDataset(Xm, Mm, y, test_idx, stream)
        tel = torch.utils.data.DataLoader(te, batch_size=128, shuffle=False, num_workers=4)
        torch.cuda.synchronize() if device == 'cuda' else None
        t0 = time.time()
        logits_t, ys_t = predict(model, tel)
        torch.cuda.synchronize() if device == 'cuda' else None
        st = scores(logits_t, ys_t)
        pd.DataFrame(dict(idx=np.asarray(test_idx), pred=logits_t.argmax(1).numpy(), true=ys_t.numpy())).to_csv(
            CACHE / ('predtest_%s_%s_seed%d.csv' % (name, tag, seed)), index=False)
        test = dict(test_top1=round(st['top1'], 4), test_top5=round(st['top5'], 4), test_macroF1=round(st['f1'], 4),
                    test_clips_per_s=round(len(test_idx) / (time.time() - t0), 1))
        print('  TEST (%s): top1 %.4f | top5 %.4f | macroF1 %.4f' % ('U4+U8' if tag == 'P1' else tag, st['top1'], st['top5'], st['f1']))
    return dict(name=name, tag=tag, seed=seed, params_M=round(n_params / 1e6, 2), minutes=round((time.time() - t_start) / 60, 1),
                last_top1=round(last['top1'], 4), last_macroF1=round(last['f1'], 4),
                best_macroF1=round(best['f1'], 4), best_epoch=best['epoch'], epochs=epochs,
                last3_top1=round(float(np.mean([e[1] for e in evals[-3:]])), 4),
                last3_macroF1=round(float(np.mean([e[2] for e in evals[-3:]])), 4),
                eval_epochs=' '.join(str(e[0]) for e in evals), **test)


def run_bilstm(name, split, seed, epochs=200, lr=3e-5, bs=64, tag='P1', eval_every=20, test_idx=None):
    """The authors' recipe: Adam lr 3e-5, batch 64, 200 epochs, categorical cross-entropy, no augmentation,
    no early stopping. fp32, as in their Keras code. The model after the last epoch is tested."""
    set_seed(seed)
    Xm = np.load(CACHE / 'X.npy', mmap_mode='r'); Mm = np.load(CACHE / 'M.npy', mmap_mode='r')
    tr = BdslDataset(Xm, Mm, y, split['train'], 'joint', aug=False)
    va = BdslDataset(Xm, Mm, y, split['val'], 'joint')
    tl = torch.utils.data.DataLoader(tr, batch_size=bs, shuffle=True, num_workers=4, drop_last=False, pin_memory=True)
    vl = torch.utils.data.DataLoader(va, batch_size=256, shuffle=False, num_workers=4)
    model = BaseBiLSTM().to(device)
    n_params = sum(p.numel() for p in model.parameters())
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    crit = nn.CrossEntropyLoss()
    print('%s %s seed %d | base-paper Bi-LSTM | %.2fM params | train %d val %d' % (name, tag, seed, n_params / 1e6, len(tr), len(va)))
    t_start, evals = time.time(), []
    for ep in range(epochs):
        model.train()
        t0, tot, n = time.time(), 0.0, 0
        for x, m, y_, _ in tl:
            x, y_ = x.to(device), y_.to(device)
            opt.zero_grad(set_to_none=True)
            loss = crit(model(x)[0], y_)
            loss.backward()
            opt.step()
            tot += loss.item() * len(y_); n += len(y_)
        if ep == 0:
            print('  first epoch took %.1f s -> about %.0f min for this run' % (time.time() - t0, (time.time() - t0) * epochs / 60))
        if (ep + 1) % eval_every == 0 or ep == epochs - 1:
            model.eval()
            with torch.no_grad():
                lv, yv = [], []
                for x, m, y_, _ in vl:
                    lv.append(model(x.to(device))[0].float().cpu()); yv.append(y_)
            sc = scores(torch.cat(lv), torch.cat(yv))
            evals.append((ep + 1, sc['top1'], sc['f1']))
            print('  ep %3d | loss %.3f | val top1 %.3f macroF1 %.3f | %.0f min so far'
                  % (ep + 1, tot / n, sc['top1'], sc['f1'], (time.time() - t_start) / 60))
    out = dict(name=name, tag=tag, seed=seed, params_M=round(n_params / 1e6, 2), minutes=round((time.time() - t_start) / 60, 1),
               epochs=epochs, last3_top1=round(float(np.mean([e[1] for e in evals[-3:]])), 4),
               last3_macroF1=round(float(np.mean([e[2] for e in evals[-3:]])), 4))
    if test_idx is not None:                      # ONE evaluation on the test signers, after training
        te = BdslDataset(Xm, Mm, y, test_idx, 'joint')
        tel = torch.utils.data.DataLoader(te, batch_size=256, shuffle=False, num_workers=4)
        model.eval()
        with torch.no_grad():
            lt, yt = [], []
            for x, m, y_, _ in tel:
                lt.append(model(x.to(device))[0].float().cpu()); yt.append(y_)
        lt, yt = torch.cat(lt), torch.cat(yt)
        st = scores(lt, yt)
        pd.DataFrame(dict(idx=np.asarray(test_idx), pred=lt.argmax(1).numpy(), true=yt.numpy())).to_csv(
            CACHE / ('predtest_%s_%s_seed%d.csv' % (name, tag, seed)), index=False)
        out.update(test_top1=round(st['top1'], 4), test_top5=round(st['top5'], 4), test_macroF1=round(st['f1'], 4))
        print('  TEST (%s): top1 %.4f | top5 %.4f | macroF1 %.4f' % (tag, st['top1'], st['top5'], st['f1']))
    return out

In [11]:
# ---- Cell 10b: quick mixed-precision check (about 1 minute; session 1 showed it is clean) ----
def amp_check(n_steps=12):
    Xm = np.load(CACHE / 'X.npy', mmap_mode='r'); Mm = np.load(CACHE / 'M.npy', mmap_mode='r')
    ds = BdslDataset(Xm, Mm, y, inner[INNER_VAL_SIGNERS[0]]['train'], 'joint', aug=True)
    dl = torch.utils.data.DataLoader(ds, batch_size=64, shuffle=True, num_workers=4, drop_last=True)
    crit = nn.CrossEntropyLoss(label_smoothing=0.1)
    bad = 0
    for kw in (None, dict(), dict(n_ma=2, masked_pool=True)):
        set_seed(0)
        model = (CompactSTGCN() if kw is None else BdslNet(**kw)).to(device).train()
        params = [p for p in model.parameters() if p.requires_grad]
        opt = torch.optim.SGD(params, lr=0.1, momentum=0.9, nesterov=True, weight_decay=1e-4)
        scaler = torch.amp.GradScaler('cuda')
        for k, (x, m, y_, _) in enumerate(dl):
            if k == n_steps:
                break
            x, m, y_ = x.to(device), m.to(device), y_.to(device)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type='cuda', dtype=torch.float16):
                logits, _ = model(x, m, keep_prob=0.9)
                loss = crit(logits, y_)
            if not torch.isfinite(loss) or not torch.isfinite(logits).all():
                bad += 1
                continue
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(params, GRAD_CLIP)
            scaler.step(opt); scaler.update()
        del model, opt
        torch.cuda.empty_cache()
    return bad

if device == 'cuda':
    n_bad = amp_check()
    USE_AMP = n_bad == 0
    print('non-finite steps: %d -> USE_AMP = %s' % (n_bad, USE_AMP))
    if not USE_AMP:
        print('WARNING: session 1 used mixed precision; fp32 results would not be comparable and are ~1.3x slower. Send me this output.')
else:
    USE_AMP = False


Attention Enabled!
Attention Enabled!
Attention Enabled!
Attention Enabled!
Attention Enabled!
Attention Enabled!
Attention Enabled!
Attention Enabled!
Attention Enabled!
Attention Enabled!


/kaggle/working/CVPR21Chal-SLR/SL-GCN/model/decouple_gcn_attn.py:118: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.eyes = nn.Parameter(torch.tensor(torch.stack(


Attention Enabled!
Attention Enabled!
Attention Enabled!
Attention Enabled!
Attention Enabled!
Attention Enabled!
Attention Enabled!
Attention Enabled!
Attention Enabled!
Attention Enabled!
non-finite steps: 0 -> USE_AMP = True


In [12]:
# ---- Cell 11: sequence models (input: per-frame 49 nodes x 3 coordinates = 147 features) ----
N_FEAT = 3 * N_NODES

class SeqBase(nn.Module):
    """Shared interface: forward(x, m, keep_prob, lam) -> (logits, None). x: (N, C, T, V)."""
    def __init__(self):
        super().__init__()
        self.in_bn = nn.BatchNorm1d(N_FEAT)            # same input normalisation as the graph models' data_bn
    def frames(self, x):                               # -> (N, T, V*C), normalised per feature
        N, C, T, V = x.shape
        f = x.permute(0, 2, 3, 1).reshape(N, T, V * C)
        return self.in_bn(f.transpose(1, 2)).transpose(1, 2)

class MLPNet(SeqBase):
    def __init__(self, hidden=256, n_cls=60):
        super().__init__()
        self.net = nn.Sequential(nn.Flatten(), nn.Linear(T_OUT * N_FEAT, hidden), nn.BatchNorm1d(hidden), nn.ReLU(), nn.Dropout(0.3),
                                 nn.Linear(hidden, hidden), nn.BatchNorm1d(hidden), nn.ReLU(), nn.Dropout(0.3), nn.Linear(hidden, n_cls))
    def forward(self, x, m=None, keep_prob=0.9, lam=0.0):
        return self.net(self.frames(x)), None

class TCNBlock(nn.Module):
    def __init__(self, c, k, d, p=0.2):
        super().__init__()
        self.conv = nn.Sequential(nn.Conv1d(c, c, k, padding=d * (k // 2), dilation=d), nn.BatchNorm1d(c), nn.ReLU(), nn.Dropout(p),
                                  nn.Conv1d(c, c, 1), nn.BatchNorm1d(c))
    def forward(self, h):
        return F.relu(h + self.conv(h))

class TCNNet(SeqBase):
    def __init__(self, c=256, k=5, dilations=(1, 2, 4, 1, 2, 4), n_cls=60):
        super().__init__()
        self.inp = nn.Sequential(nn.Conv1d(N_FEAT, c, 1), nn.BatchNorm1d(c), nn.ReLU())
        self.blocks = nn.Sequential(*[TCNBlock(c, k, d) for d in dilations])
        self.drop = nn.Dropout(0.3); self.fc = nn.Linear(c, n_cls)
    def forward(self, x, m=None, keep_prob=0.9, lam=0.0):
        h = self.blocks(self.inp(self.frames(x).transpose(1, 2)))       # (N, c, T)
        return self.fc(self.drop(h.mean(-1))), None

class RNNNet(SeqBase):
    def __init__(self, cell='lstm', hidden=256, layers=2, bidirectional=False, n_cls=60):
        super().__init__()
        rnn = nn.LSTM if cell == 'lstm' else nn.GRU
        self.rnn = rnn(N_FEAT, hidden, num_layers=layers, batch_first=True, bidirectional=bidirectional, dropout=0.2)
        out = hidden * (2 if bidirectional else 1)
        self.drop = nn.Dropout(0.3); self.fc = nn.Linear(2 * out, n_cls)
    def forward(self, x, m=None, keep_prob=0.9, lam=0.0):
        h, _ = self.rnn(self.frames(x))                                  # (N, T, out)
        feat = torch.cat([h.mean(1), h.max(1).values], -1)               # mean + max over time
        return self.fc(self.drop(feat)), None

class TransformerNet(SeqBase):
    def __init__(self, d=192, layers=4, heads=4, ff=384, n_cls=60):
        super().__init__()
        self.inp = nn.Linear(N_FEAT, d)
        self.pos = nn.Parameter(torch.zeros(1, T_OUT + 1, d)); nn.init.trunc_normal_(self.pos, std=0.02)
        self.cls = nn.Parameter(torch.zeros(1, 1, d))
        layer = nn.TransformerEncoderLayer(d, heads, ff, dropout=0.1, batch_first=True, norm_first=True)
        self.enc = nn.TransformerEncoder(layer, layers)
        self.norm = nn.LayerNorm(d); self.fc = nn.Linear(d, n_cls)
    def forward(self, x, m=None, keep_prob=0.9, lam=0.0):
        h = self.inp(self.frames(x))
        h = torch.cat([self.cls.expand(len(h), -1, -1), h], 1) + self.pos
        return self.fc(self.norm(self.enc(h)[:, 0])), None

SEQ_MODELS.update({
    'MLP': MLPNet,
    'TCN': TCNNet,
    'LSTM': lambda: RNNNet('lstm', 256, 2, False),
    'BiLSTM': lambda: RNNNet('lstm', 192, 2, True),
    'GRU': lambda: RNNNet('gru', 192, 2, True),
    'Transformer': TransformerNet,
    'BiLSTM_base': BaseBiLSTM,                       # the base paper's architecture (cell 9)
})

# ---- self-tests: shapes, parameter counts, one finite training step each ----
_x = torch.randn(4, 3, T_OUT, N_NODES, device=device); _y = torch.randint(0, 60, (4,), device=device)
for _name, _ctor in SEQ_MODELS.items():
    _m = _ctor().to(device).train()
    _out = _m(_x)[0]
    assert _out.shape == (4, 60), _name
    _loss = F.cross_entropy(_out, _y); _loss.backward()
    assert torch.isfinite(_loss) and all(torch.isfinite(p.grad).all() for p in _m.parameters() if p.grad is not None), _name
    print('%-12s %.2fM parameters' % (_name, sum(p.numel() for p in _m.parameters()) / 1e6))
del _m
print('sequence-model self-tests passed')

MLP          2.49M parameters
TCN          2.42M parameters
LSTM         0.97M parameters
BiLSTM       1.46M parameters
GRU          1.11M parameters
Transformer  1.24M parameters
BiLSTM_base  1.27M parameters
sequence-model self-tests passed


/tmp/ipykernel_24/1153348901.py:59: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.enc = nn.TransformerEncoder(layer, layers)


In [13]:
# ---- Cell 12: jobs - 7 models x 2 recipes x 3 seeds on P1 (resumable, clock-guarded) ----
EPOCHS = 38               # same as every graph model
TIME_BUDGET_H = 10.5
SEEDS = (0, 1, 2)
RECIPES = {'sgd': dict(opt_name='sgd', lr=0.1), 'adamw': dict(opt_name='adamw', lr=1e-3)}
MODELS = list(SEQ_MODELS)
# order: every model and recipe at seed 0 first, so a cut-off session still covers all of them
JOBS = [(s, mname, r) for s in SEEDS for r in RECIPES for mname in MODELS]
EXPECTED_MIN = 8.0

assert set(meta.user.values[p1['test']]) == set(TEST_SIGNERS) and len(p1['test']) == 1276
res_path = CACHE / 'seq_results.csv'
if not res_path.exists():
    prev = sorted(INPUT_ROOT.rglob('seq_results.csv'), key=lambda p: p.stat().st_mtime)
    if prev:
        shutil.copy(prev[-1], res_path)
        for f in prev[-1].parent.glob('predtest_*.csv'):
            shutil.copy(f, CACHE / f.name)
        print('resumed %d finished runs from %s' % (len(pd.read_csv(res_path)), prev[-1]))
done = pd.read_csv(res_path) if res_path.exists() else pd.DataFrame(columns=['name', 'recipe', 'seed', 'minutes'])
results = done.to_dict('records')

def expected_minutes(mname):
    seen = done[done.name == mname].minutes if len(done) else []
    return float(np.max(seen)) if len(seen) else EXPECTED_MIN

print('P1: train %d | monitor U%d %d | test U4+U8 %d | %d jobs' % (len(p1['train']), P1_VAL_SIGNER, len(p1['val']), len(p1['test']), len(JOBS)))
for seed, mname, rname in JOBS:
    if len(done) and ((done.name == mname) & (done.recipe == rname) & (done.seed == seed)).any():
        continue
    used_h = (time.time() - SESSION_START) / 3600
    if used_h + expected_minutes(mname) * 1.05 / 60 > TIME_BUDGET_H:
        print('STOP before %s %s seed %d: %.1f h used. Saved results are kept.' % (mname, rname, seed, used_h))
        break
    if device == 'cuda':
        torch.cuda.reset_peak_memory_stats()
    try:
        r = run(mname, p1, seed, epochs=EPOCHS, tag='P1_' + rname, arch=mname, test_idx=p1['test'], **RECIPES[rname])
        r['status'] = 'ok'
    except RuntimeError as e:                    # e.g. a non-finite loss: record it and go on
        print('  FAILED: %s' % e)
        r = dict(name=mname, seed=seed, status='failed: %s' % str(e)[:80])
        if device == 'cuda':
            torch.cuda.empty_cache()
    r.update(recipe=rname, amp=USE_AMP,
             peak_mem_GB=round(torch.cuda.max_memory_allocated() / 1e9, 2) if device == 'cuda' else np.nan)
    results.append(r)
    done = pd.DataFrame(results)
    done.to_csv(res_path, index=False)
print('session time used: %.1f h' % ((time.time() - SESSION_START) / 3600))

P1: train 7423 | monitor U5 600 | test U4+U8 1276 | 42 jobs
MLP P1_sgd seed 0 | n_ma=0 masked_pool=False lam_max=0.00 | 2.49M params | train 7423 val 600
  first epoch took 2 s -> about 1 min for this run
  ep   5 | loss 1.357 | val top1 0.865 macroF1 0.846 | keep 0.97 lam 0.000 | 0 min so far
  ep  10 | loss 1.087 | val top1 0.893 macroF1 0.875 | keep 0.94 lam 0.000 | 0 min so far
  ep  15 | loss 1.012 | val top1 0.917 macroF1 0.904 | keep 0.91 lam 0.000 | 1 min so far
  ep  20 | loss 0.968 | val top1 0.917 macroF1 0.904 | keep 0.90 lam 0.000 | 1 min so far
  ep  25 | loss 0.928 | val top1 0.937 macroF1 0.923 | keep 0.90 lam 0.000 | 1 min so far
  ep  30 | loss 0.904 | val top1 0.923 macroF1 0.910 | keep 0.90 lam 0.000 | 1 min so far
  ep  35 | loss 0.896 | val top1 0.930 macroF1 0.916 | keep 0.90 lam 0.000 | 1 min so far
  ep  38 | loss 0.896 | val top1 0.930 macroF1 0.917 | keep 0.90 lam 0.000 | 1 min so far
  TEST (P1_sgd): top1 0.8723 | top5 0.9647 | macroF1 0.8622
TCN P1_sgd seed

/tmp/ipykernel_24/1153348901.py:59: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.enc = nn.TransformerEncoder(layer, layers)


  first epoch took 3 s -> about 2 min for this run
  ep   5 | loss 1.131 | val top1 0.900 macroF1 0.883 | keep 0.97 lam 0.000 | 0 min so far
  ep  10 | loss 0.884 | val top1 0.902 macroF1 0.890 | keep 0.94 lam 0.000 | 1 min so far
  ep  15 | loss 0.807 | val top1 0.913 macroF1 0.906 | keep 0.91 lam 0.000 | 1 min so far
  ep  20 | loss 0.775 | val top1 0.927 macroF1 0.918 | keep 0.90 lam 0.000 | 1 min so far
  ep  25 | loss 0.758 | val top1 0.922 macroF1 0.918 | keep 0.90 lam 0.000 | 1 min so far
  ep  30 | loss 0.747 | val top1 0.933 macroF1 0.928 | keep 0.90 lam 0.000 | 2 min so far
  ep  35 | loss 0.745 | val top1 0.930 macroF1 0.923 | keep 0.90 lam 0.000 | 2 min so far
  ep  38 | loss 0.744 | val top1 0.933 macroF1 0.928 | keep 0.90 lam 0.000 | 2 min so far
  TEST (P1_sgd): top1 0.8770 | top5 0.9718 | macroF1 0.8709
BiLSTM_base P1_sgd seed 0 | n_ma=0 masked_pool=False lam_max=0.00 | 1.27M params | train 7423 val 600
  first epoch took 3 s -> about 2 min for this run
  ep   5 | loss 

/tmp/ipykernel_24/1153348901.py:59: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.enc = nn.TransformerEncoder(layer, layers)


  first epoch took 3 s -> about 2 min for this run
  ep   5 | loss 1.093 | val top1 0.892 macroF1 0.876 | keep 0.97 lam 0.000 | 0 min so far
  ep  10 | loss 0.870 | val top1 0.903 macroF1 0.894 | keep 0.94 lam 0.000 | 1 min so far
  ep  15 | loss 0.799 | val top1 0.920 macroF1 0.915 | keep 0.91 lam 0.000 | 1 min so far
  ep  20 | loss 0.768 | val top1 0.912 macroF1 0.901 | keep 0.90 lam 0.000 | 1 min so far
  ep  25 | loss 0.746 | val top1 0.927 macroF1 0.917 | keep 0.90 lam 0.000 | 1 min so far
  ep  30 | loss 0.736 | val top1 0.938 macroF1 0.933 | keep 0.90 lam 0.000 | 2 min so far
  ep  35 | loss 0.733 | val top1 0.938 macroF1 0.934 | keep 0.90 lam 0.000 | 2 min so far
  ep  38 | loss 0.732 | val top1 0.947 macroF1 0.944 | keep 0.90 lam 0.000 | 2 min so far
  TEST (P1_adamw): top1 0.8817 | top5 0.9647 | macroF1 0.8740
BiLSTM_base P1_adamw seed 0 | n_ma=0 masked_pool=False lam_max=0.00 | 1.27M params | train 7423 val 600
  first epoch took 2 s -> about 2 min for this run
  ep   5 | l

/tmp/ipykernel_24/1153348901.py:59: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.enc = nn.TransformerEncoder(layer, layers)


... (3 outputs left out)

/tmp/ipykernel_24/1153348901.py:59: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.enc = nn.TransformerEncoder(layer, layers)


  first epoch took 3 s -> about 2 min for this run
  ep   5 | loss 1.107 | val top1 0.838 macroF1 0.819 | keep 0.97 lam 0.000 | 0 min so far
  ep  10 | loss 0.882 | val top1 0.895 macroF1 0.883 | keep 0.94 lam 0.000 | 1 min so far
  ep  15 | loss 0.802 | val top1 0.918 macroF1 0.912 | keep 0.91 lam 0.000 | 1 min so far
  ep  20 | loss 0.780 | val top1 0.928 macroF1 0.921 | keep 0.90 lam 0.000 | 1 min so far
  ep  25 | loss 0.754 | val top1 0.927 macroF1 0.920 | keep 0.90 lam 0.000 | 1 min so far
  ep  30 | loss 0.747 | val top1 0.927 macroF1 0.919 | keep 0.90 lam 0.000 | 2 min so far
  ep  35 | loss 0.743 | val top1 0.927 macroF1 0.920 | keep 0.90 lam 0.000 | 2 min so far
  ep  38 | loss 0.744 | val top1 0.935 macroF1 0.930 | keep 0.90 lam 0.000 | 2 min so far
  TEST (P1_sgd): top1 0.8723 | top5 0.9671 | macroF1 0.8643
BiLSTM_base P1_sgd seed 2 | n_ma=0 masked_pool=False lam_max=0.00 | 1.27M params | train 7423 val 600
  first epoch took 2 s -> about 1 min for this run
  ep   5 | loss 

/tmp/ipykernel_24/1153348901.py:59: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.enc = nn.TransformerEncoder(layer, layers)


  first epoch took 3 s -> about 2 min for this run
  ep   5 | loss 1.052 | val top1 0.863 macroF1 0.843 | keep 0.97 lam 0.000 | 0 min so far
  ep  10 | loss 0.877 | val top1 0.910 macroF1 0.904 | keep 0.94 lam 0.000 | 1 min so far
  ep  15 | loss 0.798 | val top1 0.917 macroF1 0.908 | keep 0.91 lam 0.000 | 1 min so far
  ep  20 | loss 0.771 | val top1 0.913 macroF1 0.905 | keep 0.90 lam 0.000 | 1 min so far
  ep  25 | loss 0.743 | val top1 0.933 macroF1 0.926 | keep 0.90 lam 0.000 | 1 min so far
  ep  30 | loss 0.736 | val top1 0.952 macroF1 0.948 | keep 0.90 lam 0.000 | 2 min so far
  ep  35 | loss 0.732 | val top1 0.942 macroF1 0.937 | keep 0.90 lam 0.000 | 2 min so far
  ep  38 | loss 0.732 | val top1 0.945 macroF1 0.941 | keep 0.90 lam 0.000 | 2 min so far
  TEST (P1_adamw): top1 0.8738 | top5 0.9718 | macroF1 0.8659
BiLSTM_base P1_adamw seed 2 | n_ma=0 masked_pool=False lam_max=0.00 | 1.27M params | train 7423 val 600
  first epoch took 2 s -> about 1 min for this run
  ep   5 | l

In [14]:
# ---- Cell 13: results ----
BILSTM_AUTHORS_RECIPE, COMPACT_STGCN, AUTHORS_MEAN, BASE_PAPER = 79.73, 88.24, 72.3, 75.1   # earlier sessions / base paper
res = pd.read_csv(res_path)
ok = res[res.status == 'ok'].copy()
if len(res[res.status != 'ok']):
    print('failed runs:'); print(res[res.status != 'ok'][['name', 'recipe', 'seed', 'status']].to_string(index=False))

agg = ok.groupby(['name', 'recipe']).agg(runs=('seed', 'size'),
        test_top1=('test_top1', 'mean'), test_top1_std=('test_top1', 'std'), test_top5=('test_top5', 'mean'),
        test_macroF1=('test_macroF1', 'mean'), test_macroF1_std=('test_macroF1', 'std'),
        monitor_U5_last3_F1=('last3_macroF1', 'mean'), params_M=('params_M', 'first'),
        train_min=('minutes', 'mean'), clips_per_s=('test_clips_per_s', 'mean'), peak_mem_GB=('peak_mem_GB', 'max')).reset_index()
for c in ('test_top1', 'test_top1_std', 'test_top5', 'test_macroF1', 'test_macroF1_std', 'monitor_U5_last3_F1'):
    agg[c] = agg[c] * 100
print('=== full table (both recipes), test U4 + U8, mean +- std over seeds ===')
print(agg.round(2).to_string(index=False))
agg.to_csv(CACHE / 'seq_table_full.csv', index=False)

# rule fixed in advance: per model, the recipe with the higher U5 monitor score goes into the main table
main = agg.loc[agg.groupby('name').monitor_U5_last3_F1.idxmax()].sort_values('test_top1', ascending=False)
print('\n=== main table (recipe chosen on the monitored signer U5, not on the test signers) ===')
print(main[['name', 'recipe', 'runs', 'test_top1', 'test_top1_std', 'test_top5', 'test_macroF1', 'params_M',
            'train_min', 'clips_per_s', 'peak_mem_GB']].round(2).to_string(index=False))
main.to_csv(CACHE / 'seq_table_main.csv', index=False)
print('\nreference: compact ST-GCN %.1f%% | base paper Bi-LSTM + RQ %.1f%% (authors\' mean %.1f%%)' % (COMPACT_STGCN, BASE_PAPER, AUTHORS_MEAN))

# the confound fix: pipeline -> training recipe -> architecture
b = agg[(agg.name == 'BiLSTM_base') & (agg.recipe == 'sgd')]
if len(b):
    ours = float(b.test_top1.iloc[0])
    total = COMPACT_STGCN - AUTHORS_MEAN
    print('\n=== gain of the compact ST-GCN over the authors\' mean (%.1f points), split three ways ===' % total)
    print('  pipeline     (authors\' Bi-LSTM recipe, our inputs)      %5.1f -> %5.1f : %+5.1f points' % (AUTHORS_MEAN, BILSTM_AUTHORS_RECIPE, BILSTM_AUTHORS_RECIPE - AUTHORS_MEAN))
    print('  recipe       (same Bi-LSTM, graph models\' recipe + aug.) %5.1f -> %5.1f : %+5.1f points' % (BILSTM_AUTHORS_RECIPE, ours, ours - BILSTM_AUTHORS_RECIPE))
    print('  architecture (compact ST-GCN, same recipe)              %5.1f -> %5.1f : %+5.1f points' % (ours, COMPACT_STGCN, COMPACT_STGCN - ours))
    print('  Bi-LSTM + our recipe, per seed:', ok[(ok.name == 'BiLSTM_base') & (ok.recipe == 'sgd')].test_top1.round(4).tolist())

# per signer and hand for the main-table models (from the saved test predictions)
rows = []
for r in main.itertuples():
    for seed in SEEDS:
        f = CACHE / ('predtest_%s_seed%d.csv' % (r.name, seed))
        tag_f = CACHE / ('predtest_%s_P1_%s_seed%d.csv' % (r.name, r.recipe, seed))
        f = tag_f if tag_f.exists() else f
        if f.exists():
            pdf = pd.read_csv(f)
            pdf['user'] = meta.user.values[pdf.idx]; pdf['hand'] = meta.hand.values[pdf.idx]
            for (u, h), g in pdf.groupby(['user', 'hand']):
                rows.append(dict(model=r.name, seed=seed, user=u, hand=h, top1=(g.pred == g.true).mean()))
if rows:
    ph = pd.DataFrame(rows).groupby(['model', 'user', 'hand']).top1.mean().unstack('model').mul(100)
    print('\ntest top-1 by signer and hand (mean over seeds):'); print(ph.round(1).to_string())

=== full table (both recipes), test U4 + U8, mean +- std over seeds ===
       name recipe  runs  test_top1  test_top1_std  test_top5  test_macroF1  test_macroF1_std  monitor_U5_last3_F1  params_M  train_min  clips_per_s  peak_mem_GB
     BiLSTM  adamw     3      83.49           0.51      96.73         82.68              0.54                90.93      1.46       1.80      2817.87         0.28
     BiLSTM    sgd     3      86.86           0.70      97.65         86.12              0.75                92.41      1.46       1.83      2708.57         0.27
BiLSTM_base  adamw     3      85.81           1.21      96.32         85.04              1.22                91.29      1.27       1.53      3479.33         0.19
BiLSTM_base    sgd     3      88.37           0.68      97.34         87.93              0.82                91.18      1.27       1.53      3347.87         0.18
        GRU  adamw     3      86.99           0.83      97.73         86.19              1.03                93.73    